In [28]:
import requests
import re
from bs4 import BeautifulSoup
from supabase import create_client
from datetime import datetime
import os
import datetime

SUPABSE_URL = os.getenv("SUPABASE_URL")
SUPABSE_KEY = os.getenv("SUPABASE_KEY")
BASE_URL = "https://asia.pokemon-card.com/ph/event-search/search/"

ORGANIZER_MAPPING = {
  "Green Gate Hobbies": "GGH",
  "UniAqua Hobbies Ayala Malls Feliz": "UniAqua",
  "Japstore ph cards & hobbies": "Japstore",
  "Hground Hobbies and Toys Shop": "HG",
  "Madcap Gaming Philippines": "Madcap",
  "Hobby Stadium Inc.": "HS",
  "GGs Hobby & Gaming Lounge": "GGs",
  "Kachi Cafe & Collectibles": "Kachi",
  "ARKEN CardZone": "ARKEN",
  "Gonza Anime Merch and Hobby Shop": "Gonza",
  "Courtside": "CS",
  "Aqua Card Games Cavite": "AquaCavite",
  "Roll Play Game Lounge": "RP",
  "Whimsy Game Cafe": "Whimsy",
  "Tadaima Hobbies and Toy Shop": "Tadaima",
  "Magus Games Cebu": "Magus",
  "ForgeHobbyPlace SouthSide": "ForgeSouth",
  "GAMERSSHOP GAMING STORE INC": "Gamersshop",
  "Forge Hobby Place": "Forge",
  "Boarderland Card and Board Games": "Boarderland",
  "Strawhat Hobby Shop and Cafe": "Strawhat",
  "Tempest Hobby Toys Shop": "Tempest",
  "SideQuest: Cafe & Hobby Lounge": "SQ",
  "GameStart Hobby Shop": "GameStart",
}

supabase = create_client(SUPABSE_URL, SUPABSE_KEY)
session = requests.Session()

headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36'
}

start_date = "08-31-2025"
end_date = "06-30-2026"
page_no = 1
events_data = []

state = {
  "current_season": 1,
  "previous_event": None,
  "ubl_count": 0,
}


def get_event_prefix(event_name: str):
  if "Premier" in event_name:
    return "PBL"
  if "Master" in event_name:
    return "MBL"
  if "Ultra" in event_name:
    return "UBL"
  if "Great" in event_name:
    return "GBL"
  return "Unknown"

def get_division(event_name: str):
  if "Masters Category" in event_name or "Master Category" in event_name:
    return "Master"
  if "Seniors Category" in event_name or "Senior Category" in event_name:
    return "Senior"
  if "Juniors Category" in event_name or "Junior Category" in event_name:
    return "Junior"
  return "Unknown"

def get_shop_name(event_organizer: str):
  return ORGANIZER_MAPPING.get(event_organizer, event_organizer.replace(" ", ""))

def generate_event_code(event_name: str, event_organizer: str):
  global state

  prefix = get_event_prefix(event_name)
  if (prefix == "Unknown"):
    return "Unknown"

  if prefix == "MBL" or prefix == "PBL":
    return f"{prefix}-{get_division(event_name)}"
  
  # Update state before getting suffix for GBL/UBL
  if prefix == "GBL":
    if (state["previous_event"] == "UBL"):
      state["current_season"] += 1
    state["ubl_count"] = 0
  elif prefix == "UBL":
    state["ubl_count"] += 1
  state["previous_event"] = prefix

  if prefix == "GBL":
    return f"{prefix}-S{state['current_season']}-{get_shop_name(event_organizer)}"

  return f"{prefix}-S{state['current_season']}-{state['ubl_count']}"

def get_date(event_date: str):
  split = event_date.split("-")
  month, day = split
  year = 2025 if int(month) >= 9 else 2026

  return datetime.datetime(year=year, month=int(month), day=int(day))

def get_format_division(format_division: str):
  split = [x.lower() for x in format_division.split(" / ")]
  return split[0], split[1]

def get_event_time(event_time: str):
  split = event_time.split("-")
  return split[0], split[1]

events_data = []

while True:
  params = {
    "pageNo": page_no,
    "keyword": "ball",
    "startDate": start_date,
    "endDate": end_date
  }
  print(f"📄 Scraping Page {page_no}...")

  res = session.get(BASE_URL, headers=headers, params=params)
  
  if res.status_code != 200:
    print(f"❌ Failed to retrieve page {page_no}. Status: {response.status_code}")
    break

  with open("response.html", "w", encoding="utf-8") as f:
    f.write(res.text)

  soup = BeautifulSoup(res.text, 'html.parser')
  event_items = soup.select(".eventLink")

  if not event_items:
    print("🏁 No more events found. Ending scrape.")
    break

  page_data = []

  for item in event_items:
    href = item.attrs.get("href")
    match = re.search(r"/(\d+)/$", href)
    if not match:
      continue
    _id = int(match.group(1))

    name_el = item.select_one(".eventTitle")
    name = name_el.text.strip()

    # Ignore waiting list placeholders
    if "Waiting List" in name:
      continue

    organizer_el = item.select_one(".organizer")
    organizer = organizer_el.text.strip()

    code = generate_event_code(name, organizer)

    date_el = item.select_one(".eventDate")
    date = get_date(date_el.text.strip())

    format_division_el = item.select_one(".formatAndLeague")
    _format, division = get_format_division(format_division_el.text.strip())

    venue_el = item.select_one(".place")
    venue = venue_el.text.strip()

    event_time_el = item.select_one(".eventTime")
    start_time, end_time = get_event_time(event_time_el.text.strip())
    
    page_data.append({
      "id": _id,
      "name": name,
      "code": code,
      "date": date.isoformat(),
      "format": _format,
      "division": division,
      "venue": venue,
      "start_time": start_time,
      "end_time": end_time,
      "organizer": organizer,
    })
  
  page_no += 1
  events_data.extend(x for x in page_data if x not in events_data)

if events_data:
  try:
    result = supabase.table("events").upsert(events_data).execute()
    print(f"🚀 Successfully synced {len(events_data)} events to Supabase.")
  except Exception as e:
    print(f"❌ Supabase Error: {e}")
else:
  print("⚠️ No event data found. Check if selectors need updating.")





📄 Scraping Page 1...
📄 Scraping Page 2...
📄 Scraping Page 3...
📄 Scraping Page 4...
📄 Scraping Page 5...
🏁 No more events found. Ending scrape.
🚀 Successfully synced 63 events to Supabase.
